<a href="https://colab.research.google.com/github/fatahrahimi330/XST-Deepfake-Detection/blob/master/Model/Video_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# We use CNN + ViT + BiLSTM model to predict whether a video is deepfake or real.

## 1. Load the Trained Model

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CNN_ViT_BiLSTM()  # same architecture as during training
model.load_state_dict(torch.load("/content/drive/MyDrive/copy_best_model.pth", map_location=device))
model.to(device)
model.eval()  # important for evaluation!

## 2. Preprocess video into frames
B = batch size (1 if predicting 1 video)

T = number of frames (extract 10–30 frames per video, depending on GPU)

C = 3 (RGB)

H, W = image size (same as used during training, e.g., 224×224)

In [ ]:
import cv2
import numpy as np
from torchvision import transforms

def video_to_frames(video_path, num_frames=16, resize=(224, 224)):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total_frames // num_frames, 1)  # evenly sample frames

    for i in range(0, total_frames, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, resize)
        frames.append(frame)

        if len(frames) == num_frames:
            break

    cap.release()
    frames = np.stack(frames)  # shape (T, H, W, C)
    return frames

## 3. Apply the same transformations as training

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def preprocess_frames(frames, transform):
    frames_tensor = torch.stack([transform(f) for f in frames])
    frames_tensor = frames_tensor.unsqueeze(0)  # (1, T, C, H, W)
    return frames_tensor

## 4. Predict

In [ ]:
video_path = "/content/video_example.mp4"

frames = video_to_frames(video_path, num_frames=16)
x = preprocess_frames(frames, val_transform).to(device)

with torch.no_grad():
    output = model(x)
    prob = torch.sigmoid(output)
    pred = (prob > 0.5).int()

print(f"Probability of being Deepfake: {prob.item():.4f}")
print("Prediction:", "Fake" if pred.item() == 1 else "Real")